In [1]:
''' 
create test data set for fuzzy name matching
take name data base from github
mixed forenames and surnames
'''

import random
import string
import unicodedata
import pandas as pd
import pathlib
import numpy as np

# -----------------------------
# Configuration
# -----------------------------
PROJECT_ROOT = pathlib.Path.cwd()

# If VS Code/Jupyter runs from the script folder, go one level up
if PROJECT_ROOT.name.lower() in {"script", "scripts"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

FORENAME_FILE = PROJECT_ROOT / "data" / "common-forenames-by-country.csv"
SURNAME_FILE = PROJECT_ROOT / "data" / "common-surnames-by-country.csv"
OUTPUT_DIR = PROJECT_ROOT / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Forename file exists:", FORENAME_FILE.exists(), FORENAME_FILE)
print("Surname file exists:", SURNAME_FILE.exists(), SURNAME_FILE)
print("Output directory:", OUTPUT_DIR)

# Number of rows in final datasets (sampled with replacement from all possible pairs).
N_ROWS = 10000

# Reproducibility
SEED = 42
random.seed(SEED)

# Share of rows that should receive fuzziness in fuzzy dataset.
FUZZY_RATE = 0.85

Project root: /home/koonming/dev/Fuzzy-Name-PSI
Forename file exists: True /home/koonming/dev/Fuzzy-Name-PSI/data/common-forenames-by-country.csv
Surname file exists: True /home/koonming/dev/Fuzzy-Name-PSI/data/common-surnames-by-country.csv
Output directory: /home/koonming/dev/Fuzzy-Name-PSI/output


In [2]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize expected column names and validate schema."""
    col_map = {
        "country": "country",
        "localised name": "localised_name",
        "localized name": "localised_name",
        "romanised name": "romanised_name",
        "romanized name": "romanised_name",
    }

    normalized = {}
    for c in df.columns:
        key = str(c).strip().lower()
        normalized[c] = col_map.get(key, key.replace(" ", "_"))

    out = df.rename(columns=normalized).copy()
    required = {"country", "localised_name", "romanised_name"}
    missing = required - set(out.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}. "
            f"Found: {list(out.columns)}"
        )

    # Keep only required columns + clean types
    out = out[list(required)].copy()
    for c in required:
        out[c] = out[c].astype(str).str.strip()
    out = out[(out["localised_name"] != "") & (out["romanised_name"] != "")].reset_index(drop=True)
    return out


def load_or_sample(path: pathlib.Path, kind: str) -> pd.DataFrame:
    if path.exists():
        if path.suffix.lower() in {".xlsx", ".xls"}:
            df = pd.read_excel(path)
        else:
            df = pd.read_csv(path)
        return normalize_columns(df)
    else:
        raise NameError()
        
# neighbour in the keyboard layout or visually/audio similar
NEIGHBOR_MAP = {
    "a": "qwsz", "b": "vghn", "c": "xdfv", "d": "ersfcxt", "e": "rdswi",
    "f": "rtgdvcs", "g": "tyfhvbj", "h": "yugjbn", "i": "uojkl", "j": "uikhmn",
    "k": "iojlm", "l": "opk", "m": "njk", "n": "bhjm", "o": "pikl",
    "p": "ol", "q": "wa", "r": "tfde", "s": "wedxza", "t": "ygfrd",
    "u": "yihj", "v": "cfgb", "w": "qase", "x": "zsdc", "y": "uhgt", "z": "asx",
}

VOWELS = "aeiou"


def strip_diacritics(text: str) -> str:
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch)
    )

In [3]:
'''
def mutate_name_old(name: str) -> tuple[str, str]:
    """Return (mutated_name, error_type)."""
    name = str(name).strip()
    if not name:
        return name, "none"

    ops = [
        "deletion",
        "insertion",
        "substitution",
        "transposition",
        "vowel_swap",
        "diacritic_drop",
    ]
    weights = [0.16, 0.18, 0.26, 0.16, 0.14, 0.10]
    op = random.choices(ops, weights=weights, k=1)[0]

    chars = list(name)
    alpha_idx = [i for i, ch in enumerate(chars) if ch.isalpha()]
    if not alpha_idx:
        return name, "none"

    i = random.choice(alpha_idx)

    if op == "deletion" and len(chars) > 1:
        del chars[i]
        return "".join(chars), op

    if op == "insertion":
        insert_char = random.choice(string.ascii_lowercase)
        chars.insert(i, insert_char)
        return "".join(chars), op

    if op == "substitution":
        ch = chars[i]
        low = ch.lower()
        if low in NEIGHBOR_MAP:
            repl = random.choice(NEIGHBOR_MAP[low])
            chars[i] = repl.upper() if ch.isupper() else repl
        else:
            chars[i] = random.choice(string.ascii_lowercase)
        return "".join(chars), op

    if op == "transposition" and len(chars) > 1:
        j = i + 1 if i < len(chars) - 1 else i - 1
        chars[i], chars[j] = chars[j], chars[i]
        return "".join(chars), op

    if op == "vowel_swap":
        vowel_positions = [idx for idx, ch in enumerate(chars) if ch.lower() in VOWELS]
        if vowel_positions:
            j = random.choice(vowel_positions)
            ch = chars[j]
            replacement = random.choice([v for v in VOWELS if v != ch.lower()])
            chars[j] = replacement.upper() if ch.isupper() else replacement
            return "".join(chars), op

    # fallback / diacritic removal
    if op == "diacritic_drop":
        stripped = strip_diacritics(name)
        if stripped != name:
            return stripped, op

    # If chosen op had no effect, do a simple substitution fallback.
    j = random.choice(alpha_idx)
    chars[j] = random.choice(string.ascii_lowercase)
    return "".join(chars), "substitution_fallback"
'''

'\ndef mutate_name_old(name: str) -> tuple[str, str]:\n    """Return (mutated_name, error_type)."""\n    name = str(name).strip()\n    if not name:\n        return name, "none"\n\n    ops = [\n        "deletion",\n        "insertion",\n        "substitution",\n        "transposition",\n        "vowel_swap",\n        "diacritic_drop",\n    ]\n    weights = [0.16, 0.18, 0.26, 0.16, 0.14, 0.10]\n    op = random.choices(ops, weights=weights, k=1)[0]\n\n    chars = list(name)\n    alpha_idx = [i for i, ch in enumerate(chars) if ch.isalpha()]\n    if not alpha_idx:\n        return name, "none"\n\n    i = random.choice(alpha_idx)\n\n    if op == "deletion" and len(chars) > 1:\n        del chars[i]\n        return "".join(chars), op\n\n    if op == "insertion":\n        insert_char = random.choice(string.ascii_lowercase)\n        chars.insert(i, insert_char)\n        return "".join(chars), op\n\n    if op == "substitution":\n        ch = chars[i]\n        low = ch.lower()\n        if low in NE

In [4]:
def mutate_name(name: str) -> tuple[str, str]:
    """Return (mutated_name, error_type)."""
    name = str(name).strip()
    if not name:
        return name, "none"

    ops = [
        "deletion", # drop letter
        "insertion", # add letter
        "substitution", # substitute letter
        "transposition", # swap letters
        "vowel_swap", # vowel_swap
        "diacritic_drop", # drop accents
        "equivalent_forename", # equivalent forename: Charles -> Charlie
        "drop_middle", # drop middle name
        "abbrev_middle", # abbreviate middle name
        "abbrev_forename",
        "name_swap", # swap forename and surname
    ]
    # weights = [0.16, 0.18, 0.26, 0.16, 0.14, 0.10]
    weights = np.repeat(1/len(ops),len(ops))
    op = random.choices(ops, weights=weights, k=1)[0]

    chars = list(name)
    alpha_idx = [i for i, ch in enumerate(chars) if ch.isalpha()]
    if not alpha_idx:
        return name, "none"

    i = random.choice(alpha_idx)

    if op == "deletion" and len(chars) > 1:
        del chars[i]
        return "".join(chars), op

    if op == "insertion":
        insert_char = random.choice(string.ascii_lowercase)
        chars.insert(i, insert_char)
        return "".join(chars), op

    if op == "substitution":
        ch = chars[i]
        low = ch.lower()
        if low in NEIGHBOR_MAP:
            repl = random.choice(NEIGHBOR_MAP[low])
            chars[i] = repl.upper() if ch.isupper() else repl
        else:
            chars[i] = random.choice(string.ascii_lowercase)
        return "".join(chars), op

    if op == "transposition" and len(chars) > 1:
        j = i + 1 if i < len(chars) - 1 else i - 1
        chars[i], chars[j] = chars[j], chars[i]
        return "".join(chars), op

    if op == "vowel_swap":
        vowel_positions = [idx for idx, ch in enumerate(chars) if ch.lower() in VOWELS]
        if vowel_positions:
            j = random.choice(vowel_positions)
            ch = chars[j]
            replacement = random.choice([v for v in VOWELS if v != ch.lower()])
            chars[j] = replacement.upper() if ch.isupper() else replacement
            return "".join(chars), op

    # fallback / diacritic removal
    if op == "diacritic_drop":
        stripped = strip_diacritics(name)
        if stripped != name:
            return stripped, op

    if op == "equivalent_forename":
        # not sure 
        pass

    if op == "drop_middle":
        name_split = name.split()
        if len(name_split)==3:
            name = name_split[0] + " " + name_split[2]
            return name, op

    if op == "abbrev_forename":
        name_split = name.split()
        if len(name_split)==3:
            name = name_split[0][0] + ". " + name_split[1][0] + ". " + name_split[2]
            return name, op
        if len(name_split)==2:
            name = name_split[0][0] + ". " + name_split[1]
            return name, op
            

    if op == "abbrev_middle":
        name_split = name.split()
        if len(name_split)==3:
            name = name_split[0] + ". " + name_split[1][0] + ". " + name_split[2]
            return name, op

    if op == "name_swap":
        name_split = name.split()
        if len(name_split)==3:
            name = name_split[2] + " " + name_split[0] + " " + name_split[1]
            return name, op
        if len(name_split)==2:
            name = name_split[1] + " " + name_split[0]
            return name, op

    # If chosen op had no effect, do a simple substitution fallback.
    j = random.choice(alpha_idx)
    chars[j] = random.choice(string.ascii_lowercase)
    return "".join(chars), "substitution_fallback"

In [5]:
def generate_clean_dataset(surnames: pd.DataFrame, forenames: pd.DataFrame, n_rows: int) -> pd.DataFrame:
    # Cross join and then sample n_rows with replacement for scalability.
    s = surnames.rename(
        columns={
            "country": "country",
            "localised_name": "surname_localised",
            "romanised_name": "surname_romanised",
        }
    )
    f = forenames.rename(
        columns={
            "country": "country",
            "localised_name": "forename_localised",
            "romanised_name": "forename_romanised",
        }
    )

    #s["_k"] = 1
    #f["_k"] = 1
    # merge by countries
    full = s.merge(f, how="inner", on="country")

    clean = full.sample(n=n_rows, replace=True, random_state=SEED).reset_index(drop=True)

    
    # insert some middle names as well
    # the rate of having middle name
    m_rate = 0.2 
    # take  forenames as middle names
    m = f
    # m.loc[m.sample(n=(1-m_rate)*n_rows, random_state=SEED).index, {'forename_localised','forename_romanised'}] = '' 
    m = m.rename(columns={"forename_localised":"middle_name_localised","forename_romanised":"middle_name_romanised"})

    middle_name_idx = clean.sample(n=int(m_rate*n_rows), random_state=SEED).index
    
    clean_with_middle_name = clean.loc[middle_name_idx,:].merge(m, on='country', how="inner")
    # initialize empty middle name
    clean['middle_name_localised'] = ''
    clean['middle_name_romanised'] = ''
    clean.loc[middle_name_idx,:] = clean_with_middle_name

    # print(clean.columns)
    clean["full_name_localised"] = clean["forename_localised"] + " " + clean["middle_name_localised"] + " " + clean["surname_localised"]
    clean["full_name_romanised"] = clean["forename_romanised"] + " " + clean["middle_name_romanised"] + " " + clean["surname_romanised"]
    clean["record_id"] = [f"N{idx:08d}" for idx in range(1, len(clean) + 1)]
    return clean


def generate_fuzzy_dataset(clean: pd.DataFrame, fuzzy_rate: float) -> pd.DataFrame:
    fuzzy = clean.copy()

    noisy_flags = [random.random() < fuzzy_rate for _ in range(len(fuzzy))]
    mutated_names = []
    error_types = []

    for base_name, make_noisy in zip(fuzzy["full_name_romanised"].tolist(), noisy_flags):
        if make_noisy:
            mutated, op = mutate_name(base_name)
        else:
            mutated, op = base_name, "none"
        mutated_names.append(mutated)
        error_types.append(op)

    fuzzy["original_name"] = fuzzy["full_name_romanised"]
    fuzzy["fuzzy_name"] = mutated_names
    fuzzy["error_type"] = error_types
    fuzzy["is_noisy"] = [op != "none" for op in error_types]
    return fuzzy

In [6]:
def main() -> tuple[pd.DataFrame, pd.DataFrame]:
    surnames = load_or_sample(SURNAME_FILE, "surname")
    forenames = load_or_sample(FORENAME_FILE, "forename")

    clean = generate_clean_dataset(surnames, forenames, n_rows=N_ROWS)
    fuzzy = generate_fuzzy_dataset(clean, fuzzy_rate=FUZZY_RATE)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    clean_path = OUTPUT_DIR / "clean_names.csv"
    fuzzy_path = OUTPUT_DIR / "fuzzy_names.csv"

    clean.to_csv(clean_path, index=False, encoding="utf-8")
    fuzzy.to_csv(fuzzy_path, index=False, encoding="utf-8")

    print(f"Clean dataset rows: {len(clean):,} -> {clean_path}")
    print(f"Fuzzy dataset rows: {len(fuzzy):,} -> {fuzzy_path}")
    print("\nError type distribution:")
    print(fuzzy["error_type"].value_counts(dropna=False))
    return clean, fuzzy


clean_df, fuzzy_df = main()
display(clean_df.head(5))
display(fuzzy_df[["record_id", "original_name", "fuzzy_name", "error_type", "is_noisy"]].head(50))

Clean dataset rows: 10,000 -> /home/koonming/dev/Fuzzy-Name-PSI/output/clean_names.csv
Fuzzy dataset rows: 10,000 -> /home/koonming/dev/Fuzzy-Name-PSI/output/fuzzy_names.csv

Error type distribution:
error_type
substitution_fallback    2577
none                     1462
name_swap                 803
vowel_swap                803
insertion                 778
abbrev_forename           768
transposition             763
substitution              762
deletion                  754
abbrev_middle             188
diacritic_drop            185
drop_middle               157
Name: count, dtype: int64


,surname_localised,surname_romanised,country,forename_localised,forename_romanised,middle_name_localised,middle_name_romanised,full_name_localised,full_name_romanised,record_id
0,Gutiérrez,Gutiérrez,ES,Julen,Julen,Ane,Ane,Julen Ane Gutiérrez,Julen Ane Gutiérrez,N00000001
1,Богда́нов,Bogdanov,RU,Артём,Artyom,,,Артём Богда́нов,Artyom Bogdanov,N00000002
2,Əliyev,Aliyev,AZ,Huseyn,Huseyn,,,Huseyn Əliyev,Huseyn Aliyev,N00000003
3,Gutiérrez,Gutiérrez,ES,Julen,Julen,Nahia,Nahia,Julen Nahia Gutiérrez,Julen Nahia Gutiérrez,N00000004
4,Montes,Montes,BR,Maria Alice,Maria Alice,,,Maria Alice Montes,Maria Alice Montes,N00000005


,record_id,original_name,fuzzy_name,error_type,is_noisy
0,N00000001,Julen Ane Gutiérrez,Gutiérrez Julen Ane,name_swap,True
1,N00000002,Artyom Bogdanov,Atryom Bogdanov,transposition,True
2,N00000003,Huseyn Aliyev,Aliyev Huseyn,name_swap,True
3,N00000004,Julen Nahia Gutiérrez,Julen. N. Gutiérrez,abbrev_middle,True
4,N00000005,Maria Alice Montes,M. A. Montes,abbrev_forename,True
5,N00000006,Sebastián Mendoza,Mendoza Sebastián,name_swap,True
6,N00000007,Soo-ah Jun,Soo-ah Jun,none,False
7,N00000008,Isidora Reyes,psidora Reyes,substitution_fallback,True
8,N00000009,Julen Izaro Gutiérrez,Julen Izaro Gutierrez,diacritic_drop,True
9,N00000010,Chun-hung Lin,Chan-hung Lin,vowel_swap,True
